In [ ]:
!pip install gradio --quiet

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import gradio as gr
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
IMG_SIZE    = (300, 300)
BATCH_SIZE  = 32
EPOCHS_HEAD = 15
EPOCHS_FINE = 30
NUM_CLASSES = 4
DATA_DIR    = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"  # ✅ Fixed
MODEL_PATH  = "/kaggle/working/best_model.keras"

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "Training"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "Training"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, "Testing"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())
print("Classes:", CLASS_NAMES)

In [ ]:
def build_model():
    base_model = EfficientNetB3(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.6),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    return model, base_model

model, base_model = build_model()
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(MODEL_PATH, save_best_only=True, verbose=1)
]

print("\n📍 Phase 1: Training the classification head...")
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(MODEL_PATH, save_best_only=True, verbose=1)
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n📍 Phase 2: Fine-tuning the full network...")
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)

In [ ]:
def plot_history(h1, h2):
    acc      = h1.history['accuracy']     + h2.history['accuracy']
    val_acc  = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss     = h1.history['loss']         + h2.history['loss']
    val_loss = h1.history['val_loss']     + h2.history['val_loss']

    epochs = range(1, len(acc) + 1)
    phase_split = len(h1.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, train, val, title in zip(
        axes, [acc, loss], [val_acc, val_loss], ["Accuracy", "Loss"]
    ):
        ax.plot(epochs, train, label=f"Train {title}")
        ax.plot(epochs, val,   label=f"Val {title}")
        ax.axvline(phase_split, color='red', linestyle='--', label="Fine-tune start")
        ax.set_title(title)
        ax.legend()
        ax.set_xlabel("Epoch")

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_history.png", dpi=150)
    plt.show()

plot_history(history1, history2)

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)

print("\n📊 Evaluating on test set...")
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss:     {test_loss:.4f}")

y_pred_probs = model.predict(test_gen)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()

# ✅ FIXED Grad-CAM — properly routes through base model's graph
def make_gradcam_heatmap(img_array, model):
    base = model.layers[0]  # EfficientNetB3 submodel
    last_conv_layer = base.get_layer("top_conv")

    grad_model = tf.keras.Model(
        inputs=base.input,
        outputs=[last_conv_layer.output, base.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, _ = grad_model(img_array)
        predictions = model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), CLASS_NAMES[pred_index.numpy()], predictions[0].numpy()

def overlay_gradcam(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array_expanded = np.expand_dims(img_array, axis=0)
    heatmap, pred_class, probs = make_gradcam_heatmap(img_array_expanded, model)
    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    original = np.uint8(255 * img_array)
    superimposed = cv2.addWeighted(original, 0.6, heatmap_colored, 0.4, 0)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(original);     axes[0].set_title("Original MRI");  axes[0].axis('off')
    axes[1].imshow(superimposed); axes[1].set_title("Grad-CAM");       axes[1].axis('off')
    plt.suptitle(f"Prediction: {pred_class} ({np.max(probs)*100:.1f}% confidence)", fontsize=13)
    plt.tight_layout()
    plt.show()
    return pred_class, probs

In [ ]:
def diagnose(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = np.expand_dims(tf.keras.utils.img_to_array(img) / 255.0, axis=0)
    probs = model.predict(img_array, verbose=0)[0]
    confidence = np.max(probs) * 100
    predicted_class = CLASS_NAMES[np.argmax(probs)]

    if confidence < 70:
        result = "⚠️ Inconclusive — recommend specialist review"
    else:
        result = predicted_class.upper()

    print(f"\n🧠 Diagnosis:   {result}")
    print(f"   Confidence:  {confidence:.1f}%\n")
    for cls, prob in zip(CLASS_NAMES, probs):
        bar = "█" * int(prob * 30)
        print(f"   {cls:<15} {prob*100:5.1f}%  {bar}")

    return result, confidence

In [ ]:
def gradio_predict(image):
    temp_path = "/kaggle/working/temp_input.jpg"
    cv2.imwrite(temp_path, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

    img_array = np.expand_dims(
        tf.keras.utils.img_to_array(
            tf.keras.utils.load_img(temp_path, target_size=IMG_SIZE)
        ) / 255.0,
        axis=0
    )

    probs = model.predict(img_array, verbose=0)[0]
    confidence = float(np.max(probs)) * 100
    predicted_class = CLASS_NAMES[np.argmax(probs)]

    if confidence < 70:
        label = "⚠️ Inconclusive — recommend specialist review"
    else:
        label = predicted_class.replace("_", " ").title()

    result_dict = {cls.replace("_", " ").title(): float(p) for cls, p in zip(CLASS_NAMES, probs)}
    summary = f"**Diagnosis:** {label}\n\n**Confidence:** {confidence:.1f}%"

    return result_dict, summary


demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(label="Upload MRI Scan"),
    outputs=[
        gr.Label(label="Class Probabilities"),
        gr.Markdown(label="Result")
    ],
    title="🧠 Brain Tumor MRI Analyzer",
    description=(
        "Upload a brain MRI scan to detect: Glioma, Meningioma, Pituitary Tumor, or No Tumor.\n"
        "⚠️ This is a research tool and does not replace professional medical diagnosis."
    )
)

demo.launch(share=True)

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/best_model.keras')